# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faith-amanze/content-refresh-prioritization/blob/main/work/notebooks/capstone.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

In [1]:
# ── Setup: shared across every section below ──
import pandas as pd
import numpy as np
import os, json
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score

RANDOM_STATE = 42
_candidates = [
    "/workspaces/content-refresh-prioritization/data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "data/raw/content_refresh_anonymized.csv",
]
_data_path = next(p for p in _candidates if os.path.exists(p))
df = pd.read_csv(_data_path)
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(f"Question: which pages should a content team prioritize for review this cycle, out of "
      f"{len(df):,} candidates, given limited review hours?")
print(f"Decision this supports: this week's refresh queue. Metric that matches the decision: "
      f"Precision@50 (of the top 50 ranked pages, what fraction genuinely deserve review).")


Question: which pages should a content team prioritize for review this cycle, out of 30,000 candidates, given limited review hours?
Decision this supports: this week's refresh queue. Metric that matches the decision: Precision@50 (of the top 50 ranked pages, what fraction genuinely deserve review).


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [2]:
# ── Data: scope and exclusions, verified ──
n_clients = df["client_id"].nunique()
print(f"Release: FlyRank ML internship starter sample (NOT the full ~79M-row Hugging Face warehouse)")
print(f"Rows: {len(df):,} pages | Clients: {n_clients} of 104 total | Window: single 90-day trailing snapshot")
print(f"Duplicate content_id rows (grain check): {df.groupby('content_id').size().gt(1).sum()}")

excluded = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d",
            "clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d",
            "content_id", "client_id", "provider_used", "model_used"]
print(f"\nColumns explicitly excluded from features: {len(excluded)} -- see w03 notebooks for the full")
print("field-by-field rationale and the leakage test proving the exclusion was necessary, not assumed.")


Release: FlyRank ML internship starter sample (NOT the full ~79M-row Hugging Face warehouse)
Rows: 30,000 pages | Clients: 32 of 104 total | Window: single 90-day trailing snapshot
Duplicate content_id rows (grain check): 0

Columns explicitly excluded from features: 12 -- see w03 notebooks for the full
field-by-field rationale and the leakage test proving the exclusion was necessary, not assumed.


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [3]:
# ── Methodology: baseline, features, split, model ──
numeric_features = [
    "avg_position", "impressions_90d", "clicks_90d", "ctr",
    "pageviews_90d", "sessions_90d", "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d", "days_with_impressions",
    "days_with_sessions", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_position_data"] = (df["avg_position"] > 0).astype(int)
numeric_features += ["has_keyword_data", "has_word_count", "has_position_data"]
feature_cols = numeric_features + categorical_features
X, y, groups = df[feature_cols], df["is_declining_label"], df["client_id"]

# Baseline rule (w04): transparent, no training.
df["weak_position"] = (df["avg_position"] >= 11) & (df["avg_position"] > 0)
df["has_visibility"] = df["impressions_90d"] >= 500

# Grouped, client-honest split (w05/w06) -- no client appears on both sides.
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))
overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print(f"Client overlap between train/test: {len(overlap)} (must be 0)")

prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric_features),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="unknown")),
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))]), categorical_features),
])
model = Pipeline([("prep", prep), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE))])
model.fit(X.iloc[train_idx], y.iloc[train_idx])
print("Model: logistic regression, fit on grouped-split training rows only.")


Client overlap between train/test: 0 (must be 0)


Model: logistic regression, fit on grouped-split training rows only.


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [4]:
# ── Results: model vs baseline, same held-out split, same metric ──
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]
df_test = df.iloc[test_idx].copy()

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

model_scores = model.predict_proba(X_test)[:, 1]
baseline_scores = (df_test["weak_position"] * df_test["has_visibility"] * df_test["impressions_90d"]).values

results = pd.DataFrame([
    {"model": "Base rate (test set)", "precision_at_50": round(y_test.mean(), 3)},
    {"model": "Baseline rule (w04)", "precision_at_50": round(precision_at_k(y_test, baseline_scores), 3),
     "roc_auc": round(roc_auc_score(y_test, baseline_scores), 3)},
    {"model": "Logistic regression (w05)", "precision_at_50": round(precision_at_k(y_test, model_scores), 3),
     "roc_auc": round(roc_auc_score(y_test, model_scores), 3)},
])
results


,model,precision_at_50,roc_auc
0,Base rate (test set),0.517,NaN
1,Baseline rule (w04),0.480,0.486
2,Logistic regression (w05),0.740,0.600


## 5. Limitations

*What this work cannot claim.*

In [5]:
# ── Limitations, checked against the data rather than just asserted ──
print(f"Model ceiling: ROC AUC {roc_auc_score(y_test, model_scores):.3f} -- real lift over baseline "
      f"({roc_auc_score(y_test, baseline_scores):.3f}), still far from certainty.")

preds = model.predict(X_test)
fp_rate = ((preds == 1) & (y_test.values == 0)).mean()
fn_rate = ((preds == 0) & (y_test.values == 1)).mean()
print(f"False positive rate (held-out): {fp_rate:.1%} | False negative rate: {fn_rate:.1%}")

flag_test = df[df["avg_position"] > 0].groupby("trend_direction", observed=True).agg(
    avg_ctr=("ctr", "mean"), avg_position=("avg_position", "mean")).round(3)
print("\n'Declining' does not mean 'worst performer' -- trend group vs current performance:")
print(flag_test)


Model ceiling: ROC AUC 0.600 -- real lift over baseline (0.486), still far from certainty.
False positive rate (held-out): 30.5% | False negative rate: 13.2%

'Declining' does not mean 'worst performer' -- trend group vs current performance:
                 avg_ctr  avg_position
trend_direction                       
down               0.324        15.944
flat               1.340        11.533
new                2.441        20.484
stable             0.517        16.333
up                 0.566        22.513


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [6]:
# ── Ranked recommendations: the action playbook (w07), rebuilt here ──
df["decline_prob"] = model.predict_proba(X)[:, 1]
df["is_feedly"] = df["content_type"] == "feedly article"

def assign_action(row):
    if not row["has_position_data"]:
        return "NO_ACTION", "no_position_data"
    if row["is_feedly"] and row["decline_prob"] >= 0.6:
        return "FEEDLY_REVIEW_SEPARATE", "feedly_declining_needs_own_playbook"
    if row["weak_position"] and row["has_visibility"] and row["decline_prob"] >= 0.6:
        return "PRIORITY_REFRESH", "weak_position_visible_and_model_flags_decline"
    if row["weak_position"] and row["has_visibility"]:
        return "REVIEW_CTR", "weak_position_visible_baseline_signal_only"
    if row["decline_prob"] >= 0.6 and not row["has_visibility"]:
        return "MONITOR_LOW_VOLUME", "model_flags_decline_but_too_little_traffic_to_act_on"
    return "NO_ACTION", "no_strong_signal"

actions = df.apply(assign_action, axis=1, result_type="expand")
df["action"], df["reason_code"] = actions[0], actions[1]
df["action"].value_counts()


action
NO_ACTION                 17776
PRIORITY_REFRESH           4560
REVIEW_CTR                 3976
MONITOR_LOW_VOLUME         3592
FEEDLY_REVIEW_SEPARATE       96
Name: count, dtype: int64

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [7]:
# ── Artifacts the paper embeds: regenerate the exact charts + metrics JSON used in docs/index.html ──
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def first_existing_dir(candidates):
    for p in candidates:
        parent = os.path.dirname(p.rstrip("/")) or "."
        if os.path.isdir(parent):
            return p
    return candidates[-1]

docs_assets = first_existing_dir([
    "/workspaces/content-refresh-prioritization/docs/assets",
    "../../docs/assets",
    "docs/assets",
])
os.makedirs(docs_assets, exist_ok=True)

INK, MUTED, ACCENT = "#16211C", "#4E5D55", "#B8863B"
PAPER, HAIRLINE = "#F2F4F3", "#D7DCD8"
plt.rcParams.update({"font.family": "monospace", "text.color": INK, "axes.edgecolor": HAIRLINE,
    "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER})

labels = ["Base rate\n(no model)", "Baseline rule\n(w04)", "Logistic\nRegression"]
values = [round(y_test.mean(), 3), round(precision_at_k(y_test, baseline_scores), 3),
          round(precision_at_k(y_test, model_scores), 3)]
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(labels, values, color=[HAIRLINE, MUTED, ACCENT], height=0.55, zorder=3)
for i, v in enumerate(values):
    ax.text(v + 0.015, i, f"{v:.3f}", va="center", fontsize=10.5, color=INK, fontweight="bold")
ax.set_xlabel("Precision@50 (held-out, grouped by client)")
ax.set_xlim(0, 0.85)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", color=HAIRLINE, linewidth=0.7, zorder=0)
ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(os.path.join(docs_assets, "chart_model_vs_baseline_notebook_check.png"), dpi=130)
plt.close(fig)

summary = {
    "n_pages": int(len(df)),
    "n_clients": int(df["client_id"].nunique()),
    "precision_at_50_baseline": float(precision_at_k(y_test, baseline_scores)),
    "precision_at_50_model": float(precision_at_k(y_test, model_scores)),
    "roc_auc_baseline": float(roc_auc_score(y_test, baseline_scores)),
    "roc_auc_model": float(roc_auc_score(y_test, model_scores)),
    "false_positive_rate": float(fp_rate),
    "false_negative_rate": float(fn_rate),
    "action_counts": df["action"].value_counts().to_dict(),
}
with open(os.path.join(docs_assets, "..", "capstone_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Regenerated chart + capstone_summary.json -- these numbers match the deployed paper exactly.")
for k, v in summary.items():
    print(f"  {k}: {v}")


Regenerated chart + capstone_summary.json -- these numbers match the deployed paper exactly.
  n_pages: 30000
  n_clients: 32
  precision_at_50_baseline: 0.48
  precision_at_50_model: 0.74
  roc_auc_baseline: 0.4861006169909825
  roc_auc_model: 0.599594209776934
  false_positive_rate: 0.30527055516514406
  false_negative_rate: 0.13183415319747013
  action_counts: {'NO_ACTION': 17776, 'PRIORITY_REFRESH': 4560, 'REVIEW_CTR': 3976, 'MONITOR_LOW_VOLUME': 3592, 'FEEDLY_REVIEW_SEPARATE': 96}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.